# Colab runner

Runs the project on a Colab GPU from VS Code.

**Every new session:** run cells 1 and 2 first, in order.

- Code comes from GitHub (always the latest pushed version).
- Big files (`data/`, `outputs/`, `judges/`) live on Google Drive, so they survive when the session ends.

## 1. Connect Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Get the latest code and install it

Paste your GitHub repository address between the quotes on the first line (once).

In [2]:
REPO = "https://github.com/tca-codes/moral-rag-cf.git"   # e.g. "https://github.com/your-username/moral-rag-cf.git"
DRIVE_DIR = "/content/drive/MyDrive/moral-rag-cf"

import os
assert REPO, "Paste your GitHub repository address into REPO on the first line of this cell."
assert os.path.isdir("/content/drive/MyDrive"), "Google Drive is not connected. Run cell 1 first."

if not os.path.isdir("/content/moral-rag-cf/.git"):
    !git clone -q {REPO} /content/moral-rag-cf
%cd /content/moral-rag-cf
!git pull -q
!pip install -q -r requirements.txt
!pip install -q -e .

# data, outputs and judges are stored on Drive and linked into the code folder
for name in ["data", "outputs", "judges"]:
    os.makedirs(f"{DRIVE_DIR}/{name}", exist_ok=True)
    if not os.path.exists(name):
        os.symlink(f"{DRIVE_DIR}/{name}", name)

print("\nCode version in use:")
!git log -1 --format="%h  %s  (%cr)"
!nvidia-smi --query-gpu=name,memory.total --format=csv

/content/moral-rag-cf
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for mvrag (pyproject.toml) ... done

Code version in use:
ae51dcf  Add reward training for advisors  (14 hours ago)
name, memory.total [MiB]
NVIDIA L4, 23034 MiB


## 3. Speed test (40 questions, 7B model, uncompressed)

Use this on an L4 or A100. On a T4, remove `--no-4bit` and change 16 to 2.

In [3]:
!python scripts/run_agents.py --split test --n 40 --max-new-tokens 60 --model Qwen/Qwen2.5-7B-Instruct --no-4bit --questions-per-chunk 16 --out outputs/speedtest.jsonl

40 questions selected from split=test
config.json: 100% 663/663 [00:00<00:00, 3.25MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 23.7MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 114MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 116MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 159MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 100% 27.8k/27.8k [00:00<00:00, 69.0MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/3.56G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/7.42G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/11.4G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   2% 256M/15.2G [00:04<03:04, 81.2MB/s, 45.0MB/s  ]
Reconstructing (incomplete total...):  15% 2.29G/15.2G [00:11<01:07, 191MB/s,  115MB/s  ]
Reconstructing (incomplete to

In [5]:
%cd /content/moral-rag-cf
!cp --update=none outputs/speedtest.jsonl outputs/agents_test_7B_full.jsonl
!python scripts/run_agents.py --split test --n 4000 --max-new-tokens 60 --model Qwen/Qwen2.5-7B-Instruct --no-4bit --questions-per-chunk 16 --out outputs/agents_test_7B_full.jsonl
!python scripts/run_agents.py --split test --n 4000 --max-new-tokens 60 --model Qwen/Qwen2.5-7B-Instruct --no-4bit --questions-per-chunk 16 --plain --out outputs/plain_test_7B_full.jsonl
!python scripts/table1.py --plain outputs/plain_test_7B_full.jsonl --agents outputs/agents_test_7B_full.jsonl --out outputs/table1_7B_full.json
!python scripts/score_agents.py --agents outputs/agents_test_7B_full.jsonl --out outputs/judge_scores_7B_full.json

/content/moral-rag-cf
3753 questions selected from split=test
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:04<00:00, 76.17it/s]
Done. 0 new questions written to outputs/agents_test_7B_full.jsonl
3753 questions selected from split=test
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:04<00:00, 76.59it/s]
Done. 0 new questions written to outputs/plain_test_7B_full.jsonl
Loading weights: 100% 105/105 [00:00<00:00, 3364.88it/s]
Loading weights: 100% 105/105 [00:00<00:00, 3398.24it/s]
Loading weights: 100% 105/105 [00:00<00:00, 3577.19it/s]
Loading weights: 100% 105/105 [00:00<00:00, 3959.35it/s]
Loading weights: 100% 105/105 [00:00<00:00, 3016.82it/s]
questions: 3753

moral value    plain model  moral agent   change
care               79.53%       87.18%   +7.65%
fairness           85.71%       89.18%   +3.48%
loyalty            88.40%       93.50%   +5.10%
authority          78.72%     

In [7]:
%cd /content/moral-rag-cf
!python scripts/run_aggregator.py --agents outputs/agents_test_7B_full.jsonl --no-4bit --batch-size 16 --limit 200
!python scripts/eval_aggregator.py

/content/moral-rag-cf
200 distinct (question, profile) inputs; 0 rows skipped (no agent answers yet)
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:04<00:00, 77.15it/s]
  160/200  (77s)
  200/200  (97s)
Done. 200 new outputs written to outputs/aggregator_test_prompt.jsonl

Reference: human-written reply (worker_answer)   rows scored: 214
system           rouge1     rouge2     rougeL  rougeLsum
plain            0.1665     0.0243     0.1319     0.1319
best_agent       0.1482     0.0255     0.1201     0.1201
aggregator       0.1511     0.0263     0.1234     0.1234

Reference: chatbot reply (A)   rows scored: 214
system           rouge1     rouge2     rougeL  rougeLsum
plain            0.2171     0.0551     0.1705     0.1705
best_agent       0.2082     0.0494     0.1594     0.1594
aggregator       0.2112     0.0515     0.1618     0.1618

Saved to outputs/rouge_aggregator_prompt.json


In [8]:
import json
agents = {json.loads(l)["uid"]: json.loads(l) for l in open("outputs/agents_test_7B_full.jsonl")}
for line in open("outputs/aggregator_test_prompt.jsonl").readlines()[:3]:
    r = json.loads(line)
    a = agents[r["uid"]]
    print("QUESTION:", a["question"])
    print("PROFILE (care, fairness, loyalty, authority, sanctity):", r["profile"])
    for value, answer in a["answers"].items():
        print(f"  [{value}] {answer}")
    print("MODERATOR:", r["output"])
    print("=" * 80)

QUESTION: Parents of multiple children: which one do you love more and why?
PROFILE (care, fairness, loyalty, authority, sanctity): [1, 1, 1, 0, 0]
  [care] I don't love one child more than another; each one is special and important in their own way. My job as a parent is to ensure all of them feel loved and protected.
  [fairness] I believe in treating all my children equally and loving them the same, so I don't think it's fair to compare or favor one over the others.
  [loyalty] I love each of my kids equally, but if I had to choose a reason, it's because they all bring something unique to our family, and standing by them all is what matters most.
  [authority] As much as it might seem like I'm playing favorites, I don't love any of my children more than the others. Each one is unique and special in their own way, and they all receive equal love and care according to their needs and the roles they play within our family.
  [sanctity] I don't love any more than the others; each child 

In [9]:
%cd /content/moral-rag-cf
!python scripts/train_judges.py --max-train 1000000 --max-dev 100000 --judges-dir judges/full
!python scripts/table1.py --plain outputs/plain_test_7B_full.jsonl --agents outputs/agents_test_7B_full.jsonl --judges-dir judges/full --out outputs/table1_7B_fulljudges.json
!python scripts/score_agents.py --agents outputs/agents_test_7B_full.jsonl --judges-dir judges/full --out outputs/judge_scores_7B_fulljudges.json

/content/moral-rag-cf
SocialChem: 355922 rows from data/socialchem/social-chem-101/social-chem-101.v1.0.tsv
  split counts: {'train': 233501, 'dev': 29234, 'test': 29239, 'none': 3913, 'test-extra': 20050, 'dev-extra': 19985, 'analysis': 20000}
[care] train=81117 dev=11092  share labeled good=0.65
config.json: 100% 480/480 [00:00<00:00, 1.94MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 111kB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 4.69MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 43.4MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 2.74MB/s]

model.safetensors: downloading bytes:  73% 242M/331M [00:03<00:00, 159MB/s, 20.7MB/s  ]  
model.safetensors: downloading bytes:  94% 311M/331M [00:04<00:00, 58.8MB/s, 24.2MB/s  ]
model.safetensors: downloading bytes: 100% 314M/314M [00:05<00:00, 59.6MB/s, 24.6MB/s  ] ]
model.safetensors: reconstructing file: 100% 331M/331M [00:05<00:00, 62.8MB/s, 28.8MB/s  ]
Loading weights: 100% 101/101 [00:00<00:00, 5442.74it/s]
[transformer

In [10]:
import json
m = json.load(open("judges/full/metrics.json"))
print(f"{'judge':<10}{'accuracy':>10}{'macro_f1':>10}{'majority':>10}{'n_train':>9}{'n_dev':>8}")
for v, r in m.items():
    print(f"{v:<10}{r['accuracy']:>10.3f}{r['macro_f1']:>10.3f}{r['majority_baseline']:>10.3f}{r['n_train']:>9}{r['n']:>8}")
t = json.load(open("outputs/table1_7B_fulljudges.json"))
print(f"\nTable 1 with full judges ({t['n']} questions)")
for v, r in t["rows"].items():
    print(f"{v:<10} plain {r['plain']:.2%}   agent {r['agent']:.2%}   change {r['agent']-r['plain']:+.2%}")
s = json.load(open("outputs/judge_scores_7B_fulljudges.json"))
print(f"\nown judge mean {s['own_mean']:.3f}   other judges mean {s['others_mean']:.3f}")

judge       accuracy  macro_f1  majority  n_train   n_dev
care           0.915     0.904     0.655    81117   11092
fairness       0.874     0.873     0.555    32059    4316
loyalty        0.890     0.881     0.647    34254    4555
authority      0.862     0.857     0.624    19963    2643
sanctity       0.852     0.852     0.516    13196    1743

Table 1 with full judges (3753 questions)
care       plain 92.64%   agent 97.20%   change +4.56%
fairness   plain 82.06%   agent 87.01%   change +4.95%
loyalty    plain 83.34%   agent 89.59%   change +6.25%
authority  plain 78.51%   agent 87.17%   change +8.67%
sanctity   plain 81.50%   agent 88.79%   change +7.29%

own judge mean 0.900   other judges mean 0.889


In [11]:
%cd /content/moral-rag-cf
!python scripts/run_aggregator.py --agents outputs/agents_test_7B_full.jsonl --no-4bit --batch-size 16
!python scripts/eval_aggregator.py

/content/moral-rag-cf
9144 distinct (question, profile) inputs; 0 rows skipped (no agent answers yet)
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:04<00:00, 77.68it/s] 
  160/8944  (73s)
  320/8944  (147s)
  480/8944  (221s)
  640/8944  (296s)
  800/8944  (370s)
  960/8944  (443s)
  1120/8944  (516s)
  1280/8944  (592s)
  1440/8944  (669s)
  1600/8944  (744s)
  1760/8944  (821s)
  1920/8944  (900s)
  2080/8944  (977s)
  2240/8944  (1049s)
  2400/8944  (1125s)
  2560/8944  (1200s)
  2720/8944  (1271s)
  2880/8944  (1342s)
  3040/8944  (1418s)
  3200/8944  (1494s)
  3360/8944  (1574s)
  3520/8944  (1651s)
  3680/8944  (1729s)
  3840/8944  (1802s)
  4000/8944  (1878s)
  4160/8944  (1953s)
  4320/8944  (2029s)
  4480/8944  (2105s)
  4640/8944  (2182s)
  4800/8944  (2257s)
  4960/8944  (2333s)
  5120/8944  (2407s)
  5280/8944  (2484s)
  5440/8944  (2562s)
  5600/8944  (2638s)
  5760/8944  (2714s)
  5920/8944  (2789s)
  6080/8944  (2863s)

In [ ]:
%cd /content/moral-rag-cf
!python scripts/sample_candidates.py --n 32 --samples 8 --no-4bit --out-dir outputs/reward_trial
!python scripts/train_agents.py --no-4bit --reward-dir outputs/reward_trial --out-dir outputs/adapters/agents_trial
!python scripts/run_agents.py --split test --n 16 --max-new-tokens 60 --model Qwen/Qwen2.5-7B-Instruct --no-4bit --adapters-dir outputs/adapters/agents_trial --questions-per-chunk 16 --out outputs/agents_trial_rl.jsonl

In [3]:
import os
for path in ["judges/full/care", "outputs/agents_test_7B_full.jsonl", "outputs/plain_test_7B_full.jsonl"]:
    print("OK     " if os.path.exists(f"/content/moral-rag-cf/{path}") else "MISSING", path)

OK      judges/full/care
OK      outputs/agents_test_7B_full.jsonl
OK      outputs/plain_test_7B_full.jsonl


In [4]:
%cd /content/moral-rag-cf
!python scripts/sample_candidates.py --n 32 --samples 8 --no-4bit --batch-size 16 --out-dir outputs/reward_trial
!python scripts/train_agents.py --no-4bit --reward-dir outputs/reward_trial --out-dir outputs/adapters/agents_trial
!python scripts/run_agents.py --split test --n 16 --max-new-tokens 60 --model Qwen/Qwen2.5-7B-Instruct --no-4bit --adapters-dir outputs/adapters/agents_trial --questions-per-chunk 16 --out outputs/agents_trial_rl.jsonl

/content/moral-rag-cf
32 questions from split=train, 8 candidates each, values=['care', 'fairness', 'loyalty', 'authority', 'sanctity']
config.json: 100% 663/663 [00:00<00:00, 3.48MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 26.6MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 113MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 112MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 155MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 100% 27.8k/27.8k [00:00<00:00, 87.5MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/3.86G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/7.42G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/11.4G [00:00<?, ?B/s]
Reconstructing (incomplete total...):  10% 1.45G/15.2G [00:05<01:38, 140MB/s,  606MB/s  ]
Reconstructing (incom

## When you are done

Disconnect the runtime so it stops using compute units: open the Command Palette and run **Colab: Remove Server**.